# 階段 2：技術指標計算與驗證

台股擇時策略研究專案第二階段。

**目的**：在階段 1 產出的原始資料上計算 `MA50`、`RSI(3)`、`RSI(5)`，並驗證計算正確。

**本階段不產生任何交易訊號、不做任何回測。**

**沿用階段 1 的結論**（已確認，本階段不重複檢查）：

- `data/twii_raw.csv`，6,632 筆，1999-01-05 ~ 2026-01-20
- 無缺值、無重複日期、OHLC 結構合理
- 開盤價為真實成交價（Open 等於前一日 Close 僅 0.15%）
- 1999–2002 年 Volume 全為 0，本策略不使用成交量，不影響

**本階段的重點是指標計算的正確性**，特別是 RSI 的平滑方式 ——
這是最容易寫錯、而且**寫錯不會報錯**的地方。錯了不會有 exception，
只會讓後續所有訊號默默偏掉，所以必須用獨立方法交叉驗證。

**產出**：`data/twii_indicators.csv`

---
## 1. 參數定義

所有參數集中在這一格，後續 cell 一律引用變數，不出現寫死的數字。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 參數 ============
DATA_IN     = "data/twii_raw.csv"
DATA_OUT    = "data/twii_indicators.csv"
MA_LEN      = 50
RSI_FAST    = 3
RSI_SLOW    = 5
STUDY_START = "2000-01-01"   # 績效統計起點；本階段僅用於驗證暖身是否足夠

# ============ 階段 1 已確認的事實（用來確認載入的是同一份資料） ============
EXPECTED_ROWS  = 6632
EXPECTED_FIRST = "1999-01-05"
EXPECTED_LAST  = "2026-01-20"

# ============ 驗證用設定 ============
VERIFY_FROM   = "2010-06-01"   # 手算對照的起點
VERIFY_LEN    = 30             # 手算對照的天數
RSI_OVERSOLD  = 30             # 僅用於統計「RSI 低於此值的天數」，本階段不產生訊號
RSI_MID       = 50

pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams["figure.figsize"] = (14, 8)

# ============ 驗證結果收集器 ============
CHECKS = []

def record(name, passed, detail=""):
    """記錄一個驗證項目的結果，並立刻印出結論。"""
    verdict = "通過" if passed else "異常"
    CHECKS.append({"驗證項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("參數設定完成")
print(f"  MA 長度   : {MA_LEN}")
print(f"  RSI 週期  : {RSI_FAST}（快） / {RSI_SLOW}（慢）")
print(f"  輸入      : {DATA_IN}")
print(f"  輸出      : {DATA_OUT}")

參數設定完成
  MA 長度   : 50
  RSI 週期  : 3（快） / 5（慢）
  輸入      : data/twii_raw.csv
  輸出      : data/twii_indicators.csv


---
## 2. 載入資料

從階段 1 的產出檔載入，轉成 DatetimeIndex 並依日期排序。

載入後立刻比對筆數與起訖日期是否與階段 1 一致。
**若不一致就 `raise` 中斷** —— 代表拿到的不是階段 1 檢查過的那份資料，
後面所有驗證都失去意義。

In [2]:
df = pd.read_csv(DATA_IN, index_col="Date", parse_dates=True).sort_index()

actual = {
    "筆數":     len(df),
    "起始日期": f"{df.index.min():%Y-%m-%d}",
    "結束日期": f"{df.index.max():%Y-%m-%d}",
}
expected = {"筆數": EXPECTED_ROWS, "起始日期": EXPECTED_FIRST, "結束日期": EXPECTED_LAST}

load_check = pd.DataFrame({
    "階段 1 確認值": pd.Series(expected).astype(str),
    "本次載入值":    pd.Series(actual).astype(str),
})
load_check["一致"] = np.where(
    load_check["階段 1 確認值"] == load_check["本次載入值"], "是", "★ 否")
display(load_check)

if (load_check["一致"] == "★ 否").any():
    raise RuntimeError("載入的資料與階段 1 不一致，請停止並人工確認 data/twii_raw.csv")

record("資料載入一致性", True,
       f"{len(df):,} 筆，{df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}，與階段 1 完全一致")

,階段 1 確認值,本次載入值,一致
筆數,6632,6632,是
起始日期,1999-01-05,1999-01-05,是
結束日期,2026-01-20,2026-01-20,是


===> [通過] 資料載入一致性
      6,632 筆，1999-01-05 ~ 2026-01-20，與階段 1 完全一致


---
## 3. 實作 RSI（Wilder 平滑）

不使用 TA-Lib / pandas_ta，自行實作，讓計算過程可被檢視。

### Wilder 平滑 vs 簡單移動平均

RSI 由 J. Welles Wilder 於 1978 年提出。他對漲幅／跌幅的平均**不是**用簡單移動平均，
而是一種遞迴平滑：

$$\text{avg}_t = \text{avg}_{t-1} + \frac{1}{n}\left(x_t - \text{avg}_{t-1}\right)$$

整理後就是 $\text{avg}_t = (1-\alpha)\,\text{avg}_{t-1} + \alpha\,x_t$，其中 $\alpha = 1/n$ ——
也就是 pandas 的 `ewm(alpha=1/n, adjust=False)`。

兩者的差別：

| | 簡單移動平均 `rolling(n).mean()` | Wilder 平滑 `ewm(alpha=1/n, adjust=False)` |
|---|---|---|
| 權重 | 最近 n 筆等權，更早的完全不算 | 指數衰減，**所有歷史資料都有殘留影響** |
| 記憶長度 | 恰好 n 天 | 有效長度約 2n−1 天，反應較鈍 |
| 舊資料離開時 | 會有「掉尾」跳動 | 平滑衰減，無跳動 |

**這兩者算出來的 RSI 數值不同，而且 n 越小差異越明顯。**
本策略用 RSI(3)，正是差異最大的區間。寫成 `rolling` 不會報錯，
只會讓所有進場點默默偏掉 —— 所以區塊 5 會用三種方式交叉驗證。

### 邊界處理

`avg_loss == 0`（期間內連續上漲，完全沒有跌幅）時 $RS = \infty$，
數學上 RSI 應為 100。直接相除會產生 `inf`，因此明確處理成 100。

In [3]:
def wilder_rsi(close: pd.Series, n: int) -> pd.Series:
    """Wilder (1978) RSI。

    平滑方式為 avg_t = avg_{t-1} + (1/n)(x_t - avg_{t-1})，
    等價於 ewm(alpha=1/n, adjust=False)，不是簡單移動平均。

    avg_loss == 0（連續上漲）時 RSI 定義為 100。
    """
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)

    avg_gain = gain.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    avg_loss = loss.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

    with np.errstate(divide="ignore", invalid="ignore"):
        rs  = avg_gain / avg_loss
        rsi = 100 - 100 / (1 + rs)

    # 連續上漲：avg_loss == 0 -> RS 無限大 -> RSI = 100
    rsi = rsi.mask((avg_loss == 0) & (avg_gain > 0), 100.0)
    # 完全持平（漲跌幅平均皆為 0）：RS 未定義，保留 NaN 並於下一格回報筆數
    rsi = rsi.mask((avg_loss == 0) & (avg_gain == 0), np.nan)

    return rsi.rename(f"RSI{n}")

print(wilder_rsi.__doc__)

Wilder (1978) RSI。

平滑方式為 avg_t = avg_{t-1} + (1/n)(x_t - avg_{t-1})，
等價於 ewm(alpha=1/n, adjust=False)，不是簡單移動平均。

avg_loss == 0（連續上漲）時 RSI 定義為 100。



---
## 4. 計算三個指標

- `MA50` —— 收盤價的 50 日**簡單**移動平均。這裡用 SMA 是業界慣例（長期趨勢濾網一律用 SMA），不用 Wilder。
- `RSI3`  —— `wilder_rsi(close, 3)`
- `RSI5`  —— `wilder_rsi(close, 5)`

In [4]:
ind = df.copy()
ind["MA"] = ind["Close"].rolling(MA_LEN).mean()
ind[f"RSI{RSI_FAST}"] = wilder_rsi(ind["Close"], RSI_FAST)
ind[f"RSI{RSI_SLOW}"] = wilder_rsi(ind["Close"], RSI_SLOW)

IND_COLS = ["MA", f"RSI{RSI_FAST}", f"RSI{RSI_SLOW}"]

# NaN 只應出現在暖身期開頭；若中段出現 NaN 代表計算有問題
nan_report = pd.DataFrame({
    "NaN 筆數":       ind[IND_COLS].isna().sum(),
    "首個有效值日期": [ind[c].first_valid_index().strftime("%Y-%m-%d") for c in IND_COLS],
    "暖身期理論長度": [MA_LEN, RSI_FAST, RSI_SLOW],
})
nan_report["暖身後仍有 NaN"] = [
    int(ind[c].loc[ind[c].first_valid_index():].isna().sum()) for c in IND_COLS
]
display(nan_report)

mid_nan = int(nan_report["暖身後仍有 NaN"].sum())
record("指標無中段缺值", mid_nan == 0,
       "三個指標在首個有效值之後皆無 NaN"
       if mid_nan == 0
       else f"暖身期之後仍有 {mid_nan} 個 NaN（可能為價格完全持平導致 RSI 未定義），需人工確認")

,NaN 筆數,首個有效值日期,暖身期理論長度,暖身後仍有 NaN
MA,49,1999-03-24,50,0
RSI3,3,1999-01-08,3,0
RSI5,5,1999-01-12,5,0


===> [通過] 指標無中段缺值
      三個指標在首個有效值之後皆無 NaN


In [5]:
display(ind.loc[STUDY_START:].head(5)[["Close"] + IND_COLS])
display(ind.tail(5)[["Close"] + IND_COLS])

,Close,MA,RSI3,RSI5
Date,,,,
2000-01-04,"8,756.5498","7,793.7206",99.0771,95.3384
2000-01-05,"8,849.8701","7,817.3852",99.2773,96.0152
2000-01-06,"8,922.0303","7,842.7278",99.4226,96.5055
2000-01-07,"8,849.8701","7,868.5326",76.3890,83.6396
2000-01-10,"9,102.5996","7,896.9672",89.3505,89.6692


,Close,MA,RSI3,RSI5
Date,,,,
2026-01-14,"30,941.7793","28,236.4204",90.8837,87.5778
2026-01-15,"30,810.5801","28,290.3007",69.0474,76.3791
2026-01-16,"31,408.6992","28,364.1335",88.2889,86.3359
2026-01-19,"31,639.2891","28,438.9303",91.3857,88.6429
2026-01-20,"31,759.9902","28,521.1019",92.8667,89.7728


---
## 5. 驗證 RSI 實作正確性 ★ 本階段最重要的部分

三個獨立的驗證：

- **(a) 值域檢查** —— RSI 依定義必須落在 [0, 100]
- **(b) 手算對照** —— 用純 Python 迴圈重現 Wilder 遞迴定義，證明 `ewm(alpha=1/n, adjust=False)` 確實等價
- **(c) 與 SMA 版本比較** —— 證明「用哪種平滑」有實質影響，不是無關緊要的細節

### (a) 值域檢查

RSI = 100 − 100/(1+RS)，其中 RS ≥ 0，因此 RSI 必然落在 [0, 100]。
若跑出界，代表 avg_gain / avg_loss 的計算有錯（例如漲跌幅拆分寫反、出現負值）。

In [6]:
rng = pd.DataFrame({
    "最小值":   [ind[c].min() for c in IND_COLS[1:]],
    "最大值":   [ind[c].max() for c in IND_COLS[1:]],
    "有效筆數": [int(ind[c].notna().sum()) for c in IND_COLS[1:]],
}, index=IND_COLS[1:])
rng.index.name = "指標"
rng["在 [0, 100] 內"] = np.where(
    (rng["最小值"] >= 0) & (rng["最大值"] <= 100), "是", "★ 否")
display(rng)

in_range = (rng["在 [0, 100] 內"] == "是").all()
record("(a) RSI 值域", in_range,
       f"RSI{RSI_FAST} 範圍 [{ind[f'RSI{RSI_FAST}'].min():.4f}, {ind[f'RSI{RSI_FAST}'].max():.4f}]，"
       f"RSI{RSI_SLOW} 範圍 [{ind[f'RSI{RSI_SLOW}'].min():.4f}, {ind[f'RSI{RSI_SLOW}'].max():.4f}]，"
       "皆落在 [0, 100] 內"
       if in_range else "有指標超出 [0, 100]，實作有誤")

,最小值,最大值,有效筆數,"在 [0, 100] 內"
指標,,,,
RSI3,0.5208,100.0000,6629,是
RSI5,2.4357,97.1878,6627,是


===> [通過] (a) RSI 值域
      RSI3 範圍 [0.5208, 100.0000]，RSI5 範圍 [2.4357, 97.1878]，皆落在 [0, 100] 內


### (b) 手算對照

用純 Python 迴圈，照 Wilder 的遞迴定義逐日手算，與向量化版本比對。

**一個容易出錯的細節**：Wilder 平滑是遞迴的，第 t 天的值依賴第 t−1 天，
一路遞迴回序列最開頭。所以**不能只切 30 天出來單獨手算** ——
那樣的起始值（seed）與完整序列不同，兩邊必然對不起來，
會誤判成「實作有錯」。

正確做法是：**手算迴圈也從序列最開頭跑起**，只是最後拿其中 30 天出來比對。

`ewm(adjust=False)` 的行為是以第一個非 NaN 值作為 seed，之後遞迴；
`min_periods=n` 只是把前 n−1 個輸出遮成 NaN，不影響遞迴本身。
下面的迴圈完整重現這個行為。

In [7]:
def wilder_rsi_loop(close_values, n):
    """純 Python 迴圈版 Wilder RSI，用來交叉驗證向量化版本。

    刻意不使用 pandas 的 ewm，完全照 Wilder 的遞迴定義逐日推進。
    """
    out = [np.nan] * len(close_values)
    avg_gain = avg_loss = None
    count = 0                      # 已納入遞迴的有效 delta 筆數

    for i in range(1, len(close_values)):
        delta = close_values[i] - close_values[i - 1]
        gain = max(delta, 0.0)
        loss = max(-delta, 0.0)
        count += 1

        if avg_gain is None:       # 第一個有效值作為 seed（對應 adjust=False）
            avg_gain, avg_loss = gain, loss
        else:
            avg_gain += (gain - avg_gain) / n
            avg_loss += (loss - avg_loss) / n

        if count < n:              # 對應 min_periods=n
            continue

        if avg_loss == 0:
            out[i] = 100.0 if avg_gain > 0 else np.nan
        else:
            rs = avg_gain / avg_loss
            out[i] = 100 - 100 / (1 + rs)

    return np.array(out)

# 從序列最開頭跑起，確保 seed 與向量化版本一致
loop_rsi = pd.Series(wilder_rsi_loop(ind["Close"].to_numpy(), RSI_FAST), index=ind.index)
print(f"迴圈版計算完成，共 {len(loop_rsi):,} 筆，有效值 {int(loop_rsi.notna().sum()):,} 筆")

迴圈版計算完成，共 6,632 筆，有效值 6,629 筆


In [8]:
# 取一段連續 VERIFY_LEN 個交易日出來逐日比對
win = ind.loc[VERIFY_FROM:].index[:VERIFY_LEN]

cmp = pd.DataFrame({
    "Close":     ind.loc[win, "Close"],
    "向量化 ewm": ind.loc[win, f"RSI{RSI_FAST}"],
    "純迴圈手算": loop_rsi.loc[win],
})
cmp["絕對差異"] = (cmp["向量化 ewm"] - cmp["純迴圈手算"]).abs()
cmp.index = cmp.index.strftime("%Y-%m-%d")
display(cmp.style.format({"Close": "{:,.2f}", "向量化 ewm": "{:.10f}",
                          "純迴圈手算": "{:.10f}", "絕對差異": "{:.2e}"}))

,Close,向量化 ewm,純迴圈手算,絕對差異
Date,,,,
2010-06-01,"7,289.33",47.5405988390,47.5405988390,0.00e+00
2010-06-02,"7,195.71",30.7773017706,30.7773017706,0.00e+00
2010-06-03,"7,360.28",64.1287058629,64.1287058629,1.42e-14
2010-06-04,"7,344.59",59.9949691796,59.9949691796,1.42e-14
2010-06-07,"7,157.83",27.8927544902,27.8927544902,0.00e+00
2010-06-08,"7,151.99",27.2098568938,27.2098568938,1.42e-14
2010-06-09,"7,071.67",18.0784067182,18.0784067182,0.00e+00
2010-06-10,"7,181.77",51.5265037803,51.5265037803,7.11e-15
2010-06-11,"7,299.49",70.7078444098,70.7078444098,1.42e-14


In [9]:
TOL = 1e-9

max_diff_win = cmp["絕對差異"].max()
# 順帶比對全序列，不只抽樣的 30 天
diff_all = (ind[f"RSI{RSI_FAST}"] - loop_rsi).abs()
max_diff_all = diff_all.max()

record(f"(b) 手算對照（{VERIFY_LEN} 個交易日，自 {win[0]:%Y-%m-%d} 起）",
       max_diff_win < TOL,
       f"抽樣區間最大絕對差異 {max_diff_win:.3e}，小於容差 {TOL:.0e}；"
       f"全序列（{int(diff_all.notna().sum()):,} 筆）最大絕對差異 {max_diff_all:.3e}。"
       f"確認 ewm(alpha=1/{RSI_FAST}, adjust=False) 等價於 Wilder 遞迴定義")

===> [通過] (b) 手算對照（30 個交易日，自 2010-06-01 起）
      抽樣區間最大絕對差異 2.842e-14，小於容差 1e-09；全序列（6,629 筆）最大絕對差異 3.553e-14。確認 ewm(alpha=1/3, adjust=False) 等價於 Wilder 遞迴定義


### (c) 與 SMA 版本的差異

這一項**不是在驗證對錯** —— SMA 版本並沒有「錯」，它只是另一個不同的指標。

目的是量化：如果不小心把 Wilder 寫成 `rolling(n).mean()`，數值會差多少。
如果差異很小，這個細節無關緊要；如果差異很大，就必須在報告中明確說明選擇 Wilder 的理由。

In [10]:
def sma_rsi(close: pd.Series, n: int) -> pd.Series:
    """對照組：用簡單移動平均取代 Wilder 平滑的 RSI（非本策略採用）。"""
    delta = close.diff()
    avg_gain = delta.clip(lower=0).rolling(n).mean()
    avg_loss = (-delta).clip(lower=0).rolling(n).mean()
    with np.errstate(divide="ignore", invalid="ignore"):
        rsi = 100 - 100 / (1 + avg_gain / avg_loss)
    return rsi.mask((avg_loss == 0) & (avg_gain > 0), 100.0)

rsi_sma = sma_rsi(ind["Close"], RSI_FAST)

both = pd.concat([ind[f"RSI{RSI_FAST}"], rsi_sma.rename("RSI3_SMA")], axis=1).dropna()
both["絕對差異"] = (both[f"RSI{RSI_FAST}"] - both["RSI3_SMA"]).abs()

diff_stats = pd.DataFrame({
    "值": [
        f"{both['絕對差異'].mean():.3f}",
        f"{both['絕對差異'].median():.3f}",
        f"{both['絕對差異'].max():.3f}",
        f"{(both['絕對差異'] > 5).mean():.2%}",
        f"{(both['絕對差異'] > 10).mean():.2%}",
        f"{((both[f'RSI{RSI_FAST}'] < RSI_OVERSOLD) != (both['RSI3_SMA'] < RSI_OVERSOLD)).mean():.2%}",
    ]
}, index=[
    "平均絕對差異", "中位數絕對差異", "最大絕對差異",
    "絕對差異 > 5 的比例", "絕對差異 > 10 的比例",
    f"兩者對「RSI < {RSI_OVERSOLD}」判定不同的比例",
])
diff_stats.index.name = f"Wilder RSI{RSI_FAST} vs SMA RSI{RSI_FAST}"
display(diff_stats)

,值
Wilder RSI3 vs SMA RSI3,
平均絕對差異,13.173
中位數絕對差異,11.220
最大絕對差異,82.891
絕對差異 > 5 的比例,77.93%
絕對差異 > 10 的比例,55.12%
兩者對「RSI < 30」判定不同的比例,11.99%


In [11]:
seg = both.loc["2020-01-01":"2020-12-31"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(seg.index, seg[f"RSI{RSI_FAST}"], lw=1.2, color="#3b6ea5",
             label=f"Wilder RSI({RSI_FAST})  [used]")
axes[0].plot(seg.index, seg["RSI3_SMA"], lw=1.2, color="#e08a3c",
             label=f"SMA RSI({RSI_FAST})  [comparison only]")
axes[0].axhline(RSI_OVERSOLD, color="grey", ls="--", lw=0.8)
axes[0].set_ylabel("RSI")
axes[0].set_title(f"Wilder vs SMA smoothing, RSI({RSI_FAST}) — 2020")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].fill_between(seg.index, seg["絕對差異"], color="#d1495b", alpha=0.7)
axes[1].set_ylabel("|difference|")
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_2292\2521468350.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
mean_abs_diff = both["絕對差異"].mean()

# 這一項是「有無實質影響」的說明，不是對錯判定，因此固定記為通過
record("(c) Wilder vs SMA 平滑差異（說明性，非對錯判定）", True,
       f"平均絕對差異 {mean_abs_diff:.3f} 個 RSI 點，最大 {both['絕對差異'].max():.3f}；"
       f"兩者對「RSI < {RSI_OVERSOLD}」的判定有 "
       f"{((both[f'RSI{RSI_FAST}'] < RSI_OVERSOLD) != (both['RSI3_SMA'] < RSI_OVERSOLD)).mean():.2%} "
       f"的交易日不一致。差異足以改變進場點，本專案採用 Wilder 原始定義")

===> [通過] (c) Wilder vs SMA 平滑差異（說明性，非對錯判定）
      平均絕對差異 13.173 個 RSI 點，最大 82.891；兩者對「RSI < 30」的判定有 11.99% 的交易日不一致。差異足以改變進場點，本專案採用 Wilder 原始定義


---
## 6. 驗證 MA50 暖身期 ★ 直接決定研究期間能否從 2000-01-01 開始

階段 1 刻意從 1999 年開始抓資料，就是為了讓 2000 年第一個交易日就有完整的 MA50。

判定標準：**第一個非 NaN 的 MA50 必須出現在 1999-12-31 或更早。**
若晚於這個日期，代表暖身資料不足，必須回到階段 1 往前多抓，
而不是把研究期間往後挪 —— 那會改變樣本定義。

In [13]:
first_ma  = ind["MA"].first_valid_index()
deadline  = pd.Timestamp(STUDY_START) - pd.Timedelta(days=1)   # 1999-12-31
first_std = ind.loc[STUDY_START:].index[0]

warmup = pd.DataFrame({
    "值": [
        f"{ind.index[0]:%Y-%m-%d}",
        f"{first_ma:%Y-%m-%d}",
        f"{int(ind.index.get_loc(first_ma)) + 1}",
        f"{deadline:%Y-%m-%d}",
        f"{first_std:%Y-%m-%d}",
        f"{ind.loc[first_std, 'MA']:,.2f}",
        f"{int(ind.loc[STUDY_START:, 'MA'].isna().sum())}",
    ]
}, index=[
    "資料起點", f"第一個有效 MA{MA_LEN} 日期", "  └ 位於第幾個交易日",
    "要求：不得晚於", f"{STUDY_START} 後第一個交易日",
    "  └ 該日 MA50 值", f"{STUDY_START} 之後 MA{MA_LEN} 的 NaN 筆數",
])
warmup.index.name = "項目"
display(warmup)

,值
項目,
資料起點,1999-01-05
第一個有效 MA50 日期,1999-03-24
└ 位於第幾個交易日,50
要求：不得晚於,1999-12-31
2000-01-01 後第一個交易日,2000-01-04
└ 該日 MA50 值,"7,793.72"
2000-01-01 之後 MA50 的 NaN 筆數,0


In [14]:
warmup_ok = (first_ma <= deadline) and not np.isnan(ind.loc[first_std, "MA"]) \
            and int(ind.loc[STUDY_START:, "MA"].isna().sum()) == 0

record(f"MA{MA_LEN} 暖身期充足", warmup_ok,
       f"第一個有效 MA{MA_LEN} 出現在 {first_ma:%Y-%m-%d}，"
       f"早於 {deadline:%Y-%m-%d}；"
       f"{STUDY_START} 後第一個交易日 {first_std:%Y-%m-%d} 的 MA{MA_LEN} = "
       f"{ind.loc[first_std, 'MA']:,.2f}（非 NaN），"
       f"研究期間可自 {STUDY_START} 起算"
       if warmup_ok else
       f"暖身不足：第一個有效 MA{MA_LEN} 在 {first_ma:%Y-%m-%d}，"
       f"晚於 {deadline:%Y-%m-%d}，需回到階段 1 往前多抓資料")

if not warmup_ok:
    raise RuntimeError("MA50 暖身期不足，請停止並回報，不要自行調整研究期間起點。")

===> [通過] MA50 暖身期充足
      第一個有效 MA50 出現在 1999-03-24，早於 1999-12-31；2000-01-01 後第一個交易日 2000-01-04 的 MA50 = 7,793.72（非 NaN），研究期間可自 2000-01-01 起算


---
## 7. 指標敘述統計

先看三個指標的整體分布，再統計三個頻率數字。

這些數字用來**預判後續訊號的頻率是否合理** ——
例如若 RSI3 低於 30 的日子只佔 1%，那 26 年下來進場機會會非常少。

統計範圍分「全樣本（含 1999 暖身年）」與「研究期間（2000-01-01 起）」兩組，
後續階段實際使用的是研究期間那一組。

**這一格只呈現數字，不下判斷、不做任何調整建議。**

In [15]:
study = ind.loc[STUDY_START:]

desc = study[IND_COLS].describe().T
desc["skew"] = study[IND_COLS].skew()
display(desc)

,count,mean,std,min,25%,50%,75%,max,skew
MA,"6,391.0000","9,945.3954","4,863.0867","3,916.6606","6,657.4380","8,480.8673","10,833.9627","28,521.1019",1.4798
RSI3,"6,391.0000",54.0149,26.5311,0.5208,31.5904,56.2092,77.0877,99.5213,-0.1835
RSI5,"6,391.0000",53.7955,20.8655,2.4357,37.5426,55.3902,70.4822,97.1878,-0.1881


In [16]:
def freq_table(frame, label):
    close_gt_ma = (frame["Close"] > frame["MA"])
    rows = []
    for col in [f"RSI{RSI_FAST}", f"RSI{RSI_SLOW}"]:
        valid = frame[col].notna()
        hit   = (frame[col] < RSI_OVERSOLD) & valid
        rows.append({
            "統計項目": f"{col} < {RSI_OVERSOLD}",
            "天數": int(hit.sum()),
            "有效樣本天數": int(valid.sum()),
            "比例": f"{hit.sum() / valid.sum():.2%}",
        })
    valid = frame["MA"].notna()
    rows.append({
        "統計項目": f"Close > MA{MA_LEN}",
        "天數": int((close_gt_ma & valid).sum()),
        "有效樣本天數": int(valid.sum()),
        "比例": f"{(close_gt_ma & valid).sum() / valid.sum():.2%}",
    })
    out = pd.DataFrame(rows).set_index("統計項目")
    out.columns = pd.MultiIndex.from_product([[label], out.columns])
    return out

freq = pd.concat([
    freq_table(study, f"研究期間（{STUDY_START} 起）"),
    freq_table(ind,   "全樣本（含 1999 暖身年）"),
], axis=1)
display(freq)

研究期間（2000-01-01 起）                全樣本（含 1999 暖身年）               
                             天數 有效樣本天數      比例              天數 有效樣本天數      比例
統計項目                                                                         
RSI3 < 30                  1502   6391  23.50%            1555   6629  23.46%
RSI5 < 30                  1022   6391  15.99%            1058   6627  15.96%
Close > MA50               3807   6391  59.57%            3940   6583  59.85%

In [17]:
# 逐年看 RSI3 觸及 30 以下的天數，確認機會不是集中在少數幾年
by_year = study.groupby(study.index.year).apply(
    lambda g: pd.Series({
        f"RSI{RSI_FAST}<{RSI_OVERSOLD} 天數": int((g[f"RSI{RSI_FAST}"] < RSI_OVERSOLD).sum()),
        f"RSI{RSI_SLOW}<{RSI_OVERSOLD} 天數": int((g[f"RSI{RSI_SLOW}"] < RSI_OVERSOLD).sum()),
        f"Close>MA{MA_LEN} 天數":            int((g["Close"] > g["MA"]).sum()),
        "交易日數":                           len(g),
    })
)
by_year.index.name = "年份"
display(by_year)

record("指標敘述統計", True,
       f"研究期間 RSI{RSI_FAST} 低於 {RSI_OVERSOLD} 共 "
       f"{int((study[f'RSI{RSI_FAST}'] < RSI_OVERSOLD).sum()):,} 天、"
       f"RSI{RSI_SLOW} 低於 {RSI_OVERSOLD} 共 "
       f"{int((study[f'RSI{RSI_SLOW}'] < RSI_OVERSOLD).sum()):,} 天、"
       f"收盤高於 MA{MA_LEN} 共 "
       f"{int((study['Close'] > study['MA']).sum()):,} 天（僅記錄，不作判斷）")

,RSI3<30 天數,RSI5<30 天數,Close>MA50 天數,交易日數
年份,,,,
2000,86,65,60,245
2001,71,56,95,245
2002,79,59,112,248
2003,58,30,151,249
2004,58,41,140,250
2005,68,53,132,247
2006,44,35,166,247
2007,45,33,159,243
2008,95,71,68,249


===> [通過] 指標敘述統計
      研究期間 RSI3 低於 30 共 1,502 天、RSI5 低於 30 共 1,022 天、收盤高於 MA50 共 3,807 天（僅記錄，不作判斷）


---
## 8. 關鍵期間視覺檢查

四段涵蓋不同市場結構的期間，每張圖兩個 panel：

- 上：收盤價與 MA50
- 下：RSI3 與 RSI5，含 30 與 50 水平線

要用肉眼確認三件事：

1. **MA50 在大跌時確實被跌破**，而且是在跌勢中段而非事後才反應
2. **RSI3 比 RSI5 波動更劇烈**（週期越短越敏感），兩條線的相對關係合理
3. **RSI3 確實會頻繁觸及 30 以下**，不是罕見事件

圖表用英文標籤，避免中文字型問題。

In [18]:
PERIODS = [
    ("2000-01", "2002-12", "Dot-com bust (2000-2002)"),
    ("2007-06", "2009-12", "Global financial crisis (2007H2-2009)"),
    ("2019-06", "2021-06", "COVID crash and V-shaped recovery (2019H2-2021H1)"),
    ("2021-06", "2023-06", "Rate-hike cycle (2021H2-2023H1)"),
]

def plot_period(start, end, title):
    seg = ind.loc[start:end]
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1]})

    axes[0].plot(seg.index, seg["Close"], lw=1.1, color="#3b6ea5", label="Close")
    axes[0].plot(seg.index, seg["MA"], lw=1.4, color="#d1495b", label=f"MA{MA_LEN}")
    axes[0].set_ylabel("Index level")
    axes[0].set_title(title)
    axes[0].legend(loc="best")
    axes[0].grid(alpha=0.3)

    axes[1].plot(seg.index, seg[f"RSI{RSI_FAST}"], lw=1.0, color="#3b6ea5",
                 label=f"RSI({RSI_FAST})")
    axes[1].plot(seg.index, seg[f"RSI{RSI_SLOW}"], lw=1.0, color="#e08a3c",
                 label=f"RSI({RSI_SLOW})")
    axes[1].axhline(RSI_OVERSOLD, color="grey", ls="--", lw=0.9)
    axes[1].axhline(RSI_MID, color="grey", ls=":", lw=0.9)
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel("RSI")
    axes[1].set_xlabel("Date")
    axes[1].legend(loc="best", ncol=2)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

for s, e, t in PERIODS:
    plot_period(s, e, t)

C:\Users\king5\AppData\Local\Temp\ipykernel_2292\2082745554.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# 把肉眼要確認的三件事同時量化，避免只憑印象
rows = []
for s, e, t in PERIODS:
    seg = ind.loc[s:e]
    rows.append({
        "期間": t,
        "交易日數": len(seg),
        f"收盤跌破 MA{MA_LEN} 比例": f"{(seg['Close'] < seg['MA']).mean():.1%}",
        f"RSI{RSI_FAST} 標準差": round(seg[f"RSI{RSI_FAST}"].std(), 2),
        f"RSI{RSI_SLOW} 標準差": round(seg[f"RSI{RSI_SLOW}"].std(), 2),
        f"RSI{RSI_FAST}<{RSI_OVERSOLD} 天數": int((seg[f"RSI{RSI_FAST}"] < RSI_OVERSOLD).sum()),
        f"RSI{RSI_SLOW}<{RSI_OVERSOLD} 天數": int((seg[f"RSI{RSI_SLOW}"] < RSI_OVERSOLD).sum()),
    })
period_stats = pd.DataFrame(rows).set_index("期間")
display(period_stats)

# RSI3 應比 RSI5 更敏感：標準差更大、觸及 30 以下更頻繁
std_ok = (study[f"RSI{RSI_FAST}"].std() > study[f"RSI{RSI_SLOW}"].std())
hit_ok = ((study[f"RSI{RSI_FAST}"] < RSI_OVERSOLD).sum()
          > (study[f"RSI{RSI_SLOW}"] < RSI_OVERSOLD).sum())

record(f"RSI{RSI_FAST} 較 RSI{RSI_SLOW} 敏感（週期越短越敏感）", std_ok and hit_ok,
       f"研究期間 RSI{RSI_FAST} 標準差 {study[f'RSI{RSI_FAST}'].std():.2f} > "
       f"RSI{RSI_SLOW} 標準差 {study[f'RSI{RSI_SLOW}'].std():.2f}；"
       f"觸及 {RSI_OVERSOLD} 以下 "
       f"{int((study[f'RSI{RSI_FAST}'] < RSI_OVERSOLD).sum()):,} 天 > "
       f"{int((study[f'RSI{RSI_SLOW}'] < RSI_OVERSOLD).sum()):,} 天，行為符合預期")

,交易日數,收盤跌破 MA50 比例,RSI3 標準差,RSI5 標準差,RSI3<30 天數,RSI5<30 天數
期間,,,,,,
Dot-com bust (2000-2002),738,63.8%,26.8000,21.0400,236,180
Global financial crisis (2007H2-2009),642,43.9%,27.9300,22.0900,163,122
COVID crash and V-shaped recovery (2019H2-2021H1),505,23.2%,24.9900,19.4000,73,42
Rate-hike cycle (2021H2-2023H1),508,45.5%,26.5300,20.8500,133,87


===> [通過] RSI3 較 RSI5 敏感（週期越短越敏感）
      研究期間 RSI3 標準差 26.53 > RSI5 標準差 20.87；觸及 30 以下 1,502 天 > 1,022 天，行為符合預期


---
## 9. 存檔

原始 OHLCV 加上 `MA`、`RSI3`、`RSI5` 三欄，存成 `data/twii_indicators.csv`。

**保留 1999 年的暖身資料，不裁切。** 後續階段自行從 2000-01-01 起算 ——
把裁切留給下游，這裡只負責提供完整的指標序列。

In [20]:
ind.to_csv(DATA_OUT, date_format="%Y-%m-%d")

# 讀回確認：後續階段拿到的必須和記憶體中的一致
back = pd.read_csv(DATA_OUT, index_col="Date", parse_dates=True)
roundtrip_ok = (
    len(back) == len(ind)
    and list(back.columns) == list(ind.columns)
    and np.allclose(back[IND_COLS].to_numpy(), ind[IND_COLS].to_numpy(), equal_nan=True)
)

out_info = pd.DataFrame({
    "值": [
        DATA_OUT,
        f"{os.path.getsize(DATA_OUT):,} bytes",
        f"{len(back):,}",
        ", ".join(back.columns),
        f"{back.index.min():%Y-%m-%d} ~ {back.index.max():%Y-%m-%d}",
        f"{int((back.index < pd.Timestamp(STUDY_START)).sum()):,}",
    ]
}, index=["檔名", "大小", "筆數", "欄位", "期間", "1999 暖身期筆數（已保留）"])
out_info.index.name = "項目"
display(out_info)

record("存檔與讀回一致性", roundtrip_ok,
       f"{DATA_OUT} 已寫入 {len(back):,} 筆 × {len(back.columns)} 欄，讀回數值完全一致"
       if roundtrip_ok else "讀回的資料與記憶體中不一致，請停止並人工確認")

,值
項目,
檔名,data/twii_indicators.csv
大小,"888,715 bytes"
筆數,"6,632"
欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5"
期間,1999-01-05 ~ 2026-01-20
1999 暖身期筆數（已保留）,241


===> [通過] 存檔與讀回一致性
      data/twii_indicators.csv 已寫入 6,632 筆 × 8 欄，讀回數值完全一致


---
## 10. 小結

In [21]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項驗證，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,驗證項目,結果,說明
0,資料載入一致性,通過,"6,632 筆，1999-01-05 ~ 2026-01-20，與階段 1 完全一致"
1,指標無中段缺值,通過,三個指標在首個有效值之後皆無 NaN
2,(a) RSI 值域,通過,"RSI3 範圍 [0.5208, 100.0000]，RSI5 範圍 [2.4357, 97..."
3,(b) 手算對照（30 個交易日，自 2010-06-01 起）,通過,"抽樣區間最大絕對差異 2.842e-14，小於容差 1e-09；全序列（6,629 筆）最大..."
4,(c) Wilder vs SMA 平滑差異（說明性，非對錯判定）,通過,平均絕對差異 13.173 個 RSI 點，最大 82.891；兩者對「RSI < 30」的...
5,MA50 暖身期充足,通過,第一個有效 MA50 出現在 1999-03-24，早於 1999-12-31；2000-0...
6,指標敘述統計,通過,"研究期間 RSI3 低於 30 共 1,502 天、RSI5 低於 30 共 1,022 天..."
7,RSI3 較 RSI5 敏感（週期越短越敏感）,通過,研究期間 RSI3 標準差 26.53 > RSI5 標準差 20.87；觸及 30 以下 ...
8,存檔與讀回一致性,通過,"data/twii_indicators.csv 已寫入 6,632 筆 × 8 欄，讀回數..."



共 9 項驗證，通過 9 項，異常 0 項


### 本階段結論

**指標計算正確**

- `RSI3` / `RSI5` 值域完全落在 [0, 100] 內，無溢出、無 inf。
- **Wilder 平滑實作已用獨立方法驗證**：純 Python 迴圈逐日重現 Wilder 遞迴定義，
  與 `ewm(alpha=1/n, adjust=False)` 的向量化版本在全序列上差異小於 1e-9。
  這確認了本專案的 RSI 是 Wilder 原始定義，不是常見的 SMA 誤寫版本。
- `MA50` 使用簡單移動平均（長期趨勢濾網的業界慣例），首個有效值之後無中段缺值。

**平滑方式的選擇確實有實質影響**

Wilder 版與 SMA 版的 RSI(3) 差異不是可忽略的數值噪音 ——
兩者對「RSI 是否低於 30」的判定在相當比例的交易日上不一致。
換句話說，若把 Wilder 誤寫成 `rolling(3).mean()`，
後續策略的進場點會實質改變，而且程式不會報任何錯。
本專案採用 Wilder 原始定義，區塊 5(c) 的數字即為此選擇的依據。

**暖身期充足，研究期間可自 2000-01-01 起算**

第一個有效 MA50 出現在 1999 年內，早於 1999-12-31 的要求。
2000 年第一個交易日的 MA50 為有效值，`2000-01-01` 之後無任何 MA50 缺值。
階段 1 多抓一年暖身資料的決定是有效的，**不需要回頭補抓資料**。

**指標行為符合預期**

- RSI3 的標準差大於 RSI5、觸及 30 以下的天數也多於 RSI5，
  符合「週期越短越敏感」的預期。
- 四段關鍵期間（網路泡沫、金融海嘯、COVID、升息循環）的視覺檢查顯示，
  MA50 在每次大跌中都確實被跌破，且是在跌勢中段而非事後才反應。

**無異常項目。**

---

### 產出

- `data/twii_indicators.csv` —— OHLCV + MA + RSI3 + RSI5，保留 1999 暖身期，未裁切

### 下一階段

訊號判斷與狀態機（MA50 月度濾網 + RSI 回檔進場）屬於階段 3，
**本 notebook 不產生任何訊號**。請先人工確認上述驗證結果，再進行下一階段。